<a href="https://colab.research.google.com/github/DifferentiableUniverseInitiative/JaxPM/blob/main/docs/notebooks/09-Multigrid_Poisson_Solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multigrid Poisson solver

`pm_forces` solves the Poisson equation with an FFT by default. With `solver="mg"` it uses
a real-space multigrid solver instead: a V-cycle iteration on the discrete 7-point Laplacian,
followed by a 4th-order finite-difference gradient. No Fourier transform is involved, and
in a distributed run the only communication is a halo exchange between neighbouring devices.

Multigrid is iterative, so it can start from an initial guess `u0`. The potential changes
slowly between PM steps, and a solve seeded with the previous step's potential (rescaled
by the growth factor) converges in about one V-cycle. This notebook runs on a single GPU and
compares the two solvers on one density field, then within a full PM simulation.

In [ ]:
# Install JaxPM. Runs only when the package is missing (e.g. on Colab);
# an existing local (editable) install is left untouched.
try:
    import jaxpm  # noqa: F401
except ImportError:
    %pip install -q --upgrade jax
    %pip install -q jaxpm

In [ ]:
import time

import jax
import jax.numpy as jnp
import jax_cosmo as jc
import matplotlib.pyplot as plt
import numpy as np

from jaxpm.growth import growth_factor
from jaxpm.painting import paint
from jaxpm.pm import linear_field, lpt, pm_forces
from jaxpm.utils import coherence, power_spectrum

print(jax.devices())

## Initial conditions

A $256^3$ Gaussian field in a $(256\,h^{-1}\mathrm{Mpc})^3$ box. First-order LPT gives the
particle displacements at the start of the simulation, $a_0 = 0.1$, and at $a = 1$. The
$a = 1$ displacements produce a strongly clustered density, which is the harder case for the
force comparison below.

In [ ]:
mesh_shape = (256, 256, 256)
box_size = [256., 256., 256.]
a0 = 0.1

cosmo = jc.Planck15(Omega_c=0.25, sigma8=0.8)
k = jnp.logspace(-4, 1, 128)
pk = jc.power.linear_matter_power(cosmo, k)
pk_fn = lambda x: jnp.interp(x.reshape([-1]), k, pk).reshape(x.shape)

initial_conditions = jax.jit(
    lambda key: linear_field(mesh_shape, box_size, pk_fn, seed=key))(
        jax.random.PRNGKey(0))

dx0, p0, _ = jax.jit(lambda ic: lpt(cosmo, ic, a=a0, order=1))(initial_conditions)
dx1, _, _ = jax.jit(lambda ic: lpt(cosmo, ic, a=1.0, order=1))(initial_conditions)
print(dx0.shape, dx1.shape)

## FFT and multigrid forces on the same density

The particles are given as displacements from a uniform grid (`initial_particles="uniform"`).
The multigrid solver discretises the Laplacian with finite differences, which corresponds to
the FFT solver with `laplace_fd=True`. Comparing against that reference isolates the
multigrid convergence error. The default FFT solver (`laplace_fd=False`) inverts the exact
$k^2$ and differs from both by the discretisation error of the Laplacian.

In [ ]:
def forces(dx, **kwargs):
    return pm_forces(dx, mesh_shape=mesh_shape, initial_particles="uniform", **kwargs)


def rel_err(a, b):
    return float(jnp.linalg.norm(a - b) / jnp.linalg.norm(b))


f_fft = jax.jit(lambda dx: forces(dx))(dx1)
f_fft_fd = jax.jit(lambda dx: forces(dx, laplace_fd=True))(dx1)
f_mg = jax.jit(lambda dx: forces(dx, solver="mg", mg_params={"cycles": 2}))(dx1)

print(f"MG (2 cycles) vs FFT laplace_fd=True : {rel_err(f_mg, f_fft_fd):.2e}")
print(f"MG (2 cycles) vs FFT laplace_fd=False: {rel_err(f_mg, f_fft):.2e}")
print(f"FFT laplace_fd=True vs laplace_fd=False: {rel_err(f_fft_fd, f_fft):.2e}")

In [ ]:
z = mesh_shape[2] // 2
panels = [(f_fft_fd[:, :, z, 0], "FFT (laplace_fd=True)"),
          (f_mg[:, :, z, 0], "Multigrid (2 cycles)"),
          (f_mg[:, :, z, 0] - f_fft_fd[:, :, z, 0], "Multigrid - FFT")]
vmax = float(jnp.abs(f_fft_fd[:, :, z, 0]).max())

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (img, title) in zip(axes, panels):
    lim = vmax if title != "Multigrid - FFT" else float(jnp.abs(img).max())
    im = ax.imshow(np.asarray(img), cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("x-component of the force, slice z = N/2")
plt.tight_layout()
plt.show()

## Convergence with the number of V-cycles

Each V-cycle reduces the residual by a roughly constant factor. A cold start (`u0=None`)
begins from a zero potential.

In [ ]:
cycle_list = [1, 2, 3, 4, 6]
cold_err = []
for c in cycle_list:
    f_c = jax.jit(lambda dx, c=c: forces(dx, solver="mg", mg_params={"cycles": c}))(dx1)
    cold_err.append(rel_err(f_c, f_fft_fd))
    print(f"cycles={c}: relative force error {cold_err[-1]:.2e}")

In [ ]:
plt.figure(figsize=(5, 4))
plt.semilogy(cycle_list, cold_err, "o-")
plt.xlabel("V-cycles")
plt.ylabel("relative force error vs FFT (laplace_fd=True)")
plt.title("Multigrid convergence, cold start")
plt.grid(True, which="both", alpha=0.3)
plt.show()

## Warm-starting inside a PM simulation

Three runs from the same initial conditions with 20 fixed kick-drift steps from $a_0 = 0.1$
to $a = 1$, differing only in the Poisson solver:

- FFT with `laplace_fd=True`;
- multigrid with a cold start and 2 V-cycles per step;
- multigrid with 1 V-cycle per step, warm-started from the previous step's potential.
  The scaling $D(a_i)/D(a_{i-1})$ follows the linear growth of the potential.

`return_potential=True` makes `pm_forces` also return the potential, which is carried
through the `lax.scan` loop. The first warm-started step is seeded with a converged
6-cycle solve at $a_0$.

In [ ]:
nsteps = 20
a_steps = jnp.linspace(a0, 1.0, nsteps + 1)
D = lambda a: growth_factor(cosmo, jnp.atleast_1d(a))[0]
E = lambda a: jnp.sqrt(jc.background.Esqr(cosmo, jnp.atleast_1d(a))[0])


def make_run(solver, cycles=2, warm=False):

    def step(carry, i):
        dx, p, phi = carry
        a, a_next = a_steps[i], a_steps[i + 1]
        if solver == "fft":
            f = forces(dx, laplace_fd=True)
        else:
            u0 = phi * D(a) / D(a_steps[jnp.maximum(i - 1, 0)]) if warm else None
            f, phi = forces(dx, solver="mg", mg_params={"cycles": cycles},
                            u0=u0, return_potential=True)
        f = f * 1.5 * cosmo.Omega_m
        p = p + (a_next - a) * f / (a**2 * E(a))  # kick
        dx = dx + (a_next - a) * p / (a**3 * E(a))  # drift
        return (dx, p, phi), None

    return jax.jit(lambda dx, p, phi: jax.lax.scan(step, (dx, p, phi),
                                                   jnp.arange(nsteps))[0][0])


_, phi_seed = jax.jit(lambda dx: forces(dx, solver="mg", mg_params={"cycles": 6},
                                        return_potential=True))(dx0)
phi_zero = jnp.zeros(mesh_shape, dtype=jnp.float32)

runs = {"FFT": (make_run("fft"), phi_zero),
        "MG cold, 2 cycles": (make_run("mg", cycles=2), phi_zero),
        "MG warm, 1 cycle": (make_run("mg", cycles=1, warm=True), phi_seed)}

final_dx, sim_time = {}, {}
for name, (run, phi_init) in runs.items():
    jax.block_until_ready(run(dx0, p0, phi_init))  # compile
    t0 = time.perf_counter()
    final_dx[name] = jax.block_until_ready(run(dx0, p0, phi_init))
    sim_time[name] = time.perf_counter() - t0
    print(f"{name:18s}: {sim_time[name]:.2f} s for {nsteps} steps")

density = {name: paint(dx, initial_particles="uniform") for name, dx in final_dx.items()}

In [ ]:
names = list(density)
ref = density["FFT"]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name in zip(axes, names):
    ax.imshow(np.log10(np.asarray(density[name][:, :, :16].sum(axis=-1)) + 1), cmap="magma")
    ax.set_title(name)
    ax.axis("off")
diff = np.asarray((density["MG warm, 1 cycle"] - ref)[:, :, :16].sum(axis=-1))
lim = np.abs(diff).max()
axes[3].imshow(diff, cmap="RdBu_r", vmin=-lim, vmax=lim)
axes[3].set_title("MG warm - FFT")
axes[3].axis("off")
plt.tight_layout()
plt.show()

k_ref, pk_ref = power_spectrum(ref, box_shape=box_size)
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4))
for name in names[1:]:
    k_, pk_ = power_spectrum(density[name], box_shape=box_size)
    _, r_ = coherence(density[name], ref, box_shape=box_size)
    ax0.semilogx(k_, pk_ / pk_ref, label=name)
    ax1.semilogx(k_, 1 - r_, label=name)
ax0.axhline(1, color="k", lw=0.8)
ax0.set_xlabel(r"$k$ [$h$/Mpc]")
ax0.set_ylabel(r"$P(k) / P_{\rm FFT}(k)$")
ax1.set_yscale("log")
ax1.set_xlabel(r"$k$ [$h$/Mpc]")
ax1.set_ylabel(r"$1 - r(k)$ with FFT")
ax0.legend()
plt.tight_layout()
plt.show()

## Timing

Wall-clock time of a single force evaluation at $a = 1$, after compilation, for the FFT
solver and the two multigrid settings used above. The warm-started call receives the
converged potential at $a = 1$ as `u0`.

In [ ]:
fn = {
    "FFT": jax.jit(lambda dx: forces(dx, laplace_fd=True)),
    "MG cold, 2 cycles": jax.jit(lambda dx: forces(dx, solver="mg", mg_params={"cycles": 2})),
    "MG warm, 1 cycle": jax.jit(lambda dx, u0: forces(dx, solver="mg",
                                                      mg_params={"cycles": 1}, u0=u0)),
}
_, phi1 = jax.jit(lambda dx: forces(dx, solver="mg", mg_params={"cycles": 6},
                                    return_potential=True))(dx1)
args = {"FFT": (dx1, ), "MG cold, 2 cycles": (dx1, ), "MG warm, 1 cycle": (dx1, phi1)}

force_time = {}
for name, f in fn.items():
    jax.block_until_ready(f(*args[name]))  # compile
    t = []
    for _ in range(10):
        t0 = time.perf_counter()
        jax.block_until_ready(f(*args[name]))
        t.append(time.perf_counter() - t0)
    force_time[name] = np.median(t)
    print(f"{name:18s}: {1e3 * force_time[name]:.1f} ms per force evaluation")

In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4))
ax0.bar(list(force_time), [1e3 * v for v in force_time.values()], color=["C0", "C1", "C2"])
ax0.set_ylabel("ms per force evaluation")
ax1.bar(list(sim_time), list(sim_time.values()), color=["C0", "C1", "C2"])
ax1.set_ylabel(f"s for {nsteps} PM steps")
for ax in (ax0, ax1):
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()